# Nível 1 — Dados e primeira análise com LLM
Desafio Técnico — Estágio em Engenharia de IA

Este notebook cobre a Parte A (tratamento de dados e regras determinísticas em pandas)
e a Parte B (análise com LLM sobre um cliente sinalizado).

In [1]:
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
with open("../dados/dados_nivel_1.json", encoding="utf-8") as f:
    dados = json.load(f)

taxa_cambio_usd_brl = dados["taxa_cambio_usd_brl"]
df = pd.DataFrame(dados["operacoes"])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio_usd_brl}")
print(f"Total de operações carregadas: {len(df)}")
df.head(10)

Taxa de câmbio USD/BRL: 5.4
Total de operações carregadas: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [3]:
print("=== Diagnóstico de qualidade dos dados ===\n")

print("Valores nulos por coluna:")
print(df.isnull().sum())

print("\nLinhas totalmente duplicadas:")
print(df[df.duplicated(keep=False)])

print("\nMoedas presentes:", df["moeda"].unique().tolist())

print("\nOperações com valor <= 0:", (df["valor"] <= 0).sum())

=== Diagnóstico de qualidade dos dados ===

Valores nulos por coluna:
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

Linhas totalmente duplicadas:
        id cliente_id        data  valor moeda canal                   tipo          contraparte observacao
6  OP-0007    CLI-A-3  2026-03-05  17200   BRL   pix  transferencia_enviada  Epsilon Consultoria           
9  OP-0007    CLI-A-3  2026-03-05  17200   BRL   pix  transferencia_enviada  Epsilon Consultoria           

Moedas presentes: ['BRL', 'USD']

Operações com valor <= 0: 0


## 1. Limpeza dos dados

Três problemas encontrados no diagnóstico acima:

1. **Linha duplicada** — `OP-0007` aparece duas vezes com campos idênticos.
   Decisão: manter apenas a primeira ocorrência (`drop_duplicates`).

2. **Data nula** — `OP-0017` tem `data: null` (observação diz "data não capturada pelo sistema").
   Decisão: manter a operação (o valor e os demais campos são válidos), mas registrar
   o problema. Não inventar uma data fictícia — seria introduzir informação falsa.

3. **Moeda estrangeira** — `OP-0013` está em USD. Para que somas e comparações
   façam sentido, precisamos de um valor normalizado em BRL.
   Decisão: criar coluna `valor_brl` usando a taxa de câmbio fornecida no próprio JSON.

In [4]:
# 1. Remover duplicatas exatas
antes = len(df)
df = df.drop_duplicates()
print(f"Duplicatas removidas: {antes - len(df)} linha(s)")

# 2. Registrar operações com data nula (manter, não descartar)
nulos_data = df["data"].isnull().sum()
print(f"Operações com data nula (mantidas): {nulos_data}")

# 3. Normalizar valores para BRL
df["valor_brl"] = df.apply(
    lambda row: row["valor"] * taxa_cambio_usd_brl if row["moeda"] == "USD" else row["valor"],
    axis=1
)

print(f"\nOperações após limpeza: {len(df)}")
print(df[["id", "valor", "moeda", "valor_brl"]].to_string(index=False))

Duplicatas removidas: 1 linha(s)
Operações com data nula (mantidas): 1

Operações após limpeza: 19
     id  valor moeda  valor_brl
OP-0001  18100   BRL    18100.0
OP-0002  17300   BRL    17300.0
OP-0003  18800   BRL    18800.0
OP-0004   3300   BRL     3300.0
OP-0005  25900   BRL    25900.0
OP-0006  27000   BRL    27000.0
OP-0007  17200   BRL    17200.0
OP-0008  15200   BRL    15200.0
OP-0009  16100   BRL    16100.0
OP-0010   3800   BRL     3800.0
OP-0011   5100   BRL     5100.0
OP-0012   5800   BRL     5800.0
OP-0013  12000   USD    64800.0
OP-0014   2900   BRL     2900.0
OP-0015   7000   BRL     7000.0
OP-0016   2700   BRL     2700.0
OP-0017   4300   BRL     4300.0
OP-0018   8800   BRL     8800.0
OP-0019   1400   BRL     1400.0


## 2. Agregações e regras determinísticas

Antes das regras, duas agregações pedidas no enunciado:
- Volume total transacionado por cliente (em BRL)
- Quantidade de operações por canal

In [5]:
# Volume total por cliente (em BRL)
volume_por_cliente = df.groupby("cliente_id")["valor_brl"].sum().sort_values(ascending=False)
print("Volume total transacionado por cliente (BRL):")
print(volume_por_cliente.to_string())

print()

# Quantidade de operações por canal
ops_por_canal = df.groupby("canal")["id"].count().sort_values(ascending=False)
ops_por_canal.name = "n_operacoes"
print("Operações por canal:")
print(ops_por_canal.to_string())

Volume total transacionado por cliente (BRL):
cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0

Operações por canal:
canal
pix        8
ted        5
boleto     3
cartao     2
especie    1


### Regra 1 — Fracionamento

Sinalize o **cliente** que, em uma **mesma data**, realizou **3 ou mais operações**
cuja soma ultrapassa **R\$ 50.000**, sendo que **nenhuma operação isolada atinge R\$ 20.000**.

É o padrão clássico de *smurfing*: dividir um valor grande em transferências menores
para evitar disparar alertas automáticos.

In [6]:
# Regra 1 — Fracionamento
# Agrupa por cliente + data, calcula número de ops, soma e valor máximo
grupo_dia = df.groupby(["cliente_id", "data"]).agg(
    n_ops=("valor_brl", "count"),
    soma_dia=("valor_brl", "sum"),
    max_op=("valor_brl", "max"),
).reset_index()

# Aplica os três critérios simultaneamente
fracionamento = grupo_dia[
    (grupo_dia["n_ops"] >= 3) &
    (grupo_dia["soma_dia"] > 50_000) &
    (grupo_dia["max_op"] < 20_000)
]

clientes_regra1 = fracionamento["cliente_id"].unique().tolist()

# Flag no DataFrame original
df["flag_regra1_fracionamento"] = df["cliente_id"].isin(clientes_regra1)

print("Clientes sinalizados pela Regra 1 (fracionamento):", clientes_regra1)
print()
print("Detalhamento:")
print(fracionamento.to_string(index=False))

Clientes sinalizados pela Regra 1 (fracionamento): ['CLI-A-1']

Detalhamento:
cliente_id       data  n_ops  soma_dia  max_op
   CLI-A-1 2026-03-09      3   54200.0 18800.0


### Regra 2 — Valor atípico

Sinalize a **operação** cujo valor em BRL seja **superior a 5× a mediana** dos
valores daquele mesmo cliente. Aplica apenas a clientes com **4 ou mais operações**
(com poucas operações a mediana é instável e geraria falsos positivos).

In [7]:
# Regra 2 — Valor atípico (> 5× mediana do cliente, mínimo 4 ops)

# Calcular mediana por cliente
stats_cliente = df.groupby("cliente_id")["valor_brl"].agg(["median", "count"]).reset_index()
stats_cliente.columns = ["cliente_id", "mediana_brl", "n_ops"]

# Filtrar apenas clientes com 4+ operações
stats_4plus = stats_cliente[stats_cliente["n_ops"] >= 4].copy()
stats_4plus["limiar_5x"] = stats_4plus["mediana_brl"] * 5

print("Clientes elegíveis (4+ ops) e seus limiares:")
print(stats_4plus.to_string(index=False))
print()

# Merge e flag
df = df.merge(stats_4plus[["cliente_id", "mediana_brl", "limiar_5x"]], on="cliente_id", how="left")
df["flag_regra2_valor_atipico"] = df["valor_brl"] > df["limiar_5x"]

# Operações sinalizadas
flagged_r2 = df[df["flag_regra2_valor_atipico"]]
print("Operações sinalizadas pela Regra 2 (valor atípico):")
print(flagged_r2[["id", "cliente_id", "valor_brl", "mediana_brl", "limiar_5x"]].to_string(index=False))

Clientes elegíveis (4+ ops) e seus limiares:
cliente_id  mediana_brl  n_ops  limiar_5x
   CLI-A-1      17700.0      4    88500.0
   CLI-A-4       5450.0      4    27250.0
   CLI-A-5       3600.0      4    18000.0

Operações sinalizadas pela Regra 2 (valor atípico):
     id cliente_id  valor_brl  mediana_brl  limiar_5x
OP-0013    CLI-A-4    64800.0       5450.0    27250.0


## 3. Validação das regras

O enunciado pede que mostremos explicitamente que cada regra captura o caso correto
e **não** captura um caso parecido que não se enquadra.

In [8]:
# === Validação da Regra 1 (Fracionamento) ===
print("=== Validação Regra 1 ===\n")

# CASO POSITIVO: CLI-A-1 em 2026-03-09
print("✅ CASO POSITIVO — CLI-A-1 em 2026-03-09:")
caso_pos = df[(df["cliente_id"] == "CLI-A-1") & (df["data"] == "2026-03-09")]
print(caso_pos[["id", "valor_brl"]].to_string(index=False))
print(f"   Nº operações: {len(caso_pos)} (≥ 3 ✓)")
print(f"   Soma: R$ {caso_pos['valor_brl'].sum():,.2f} (> R$ 50.000 ✓)")
print(f"   Maior operação: R$ {caso_pos['valor_brl'].max():,.2f} (< R$ 20.000 ✓)")
print(f"   → Sinalizado: SIM\n")

# CASO NEGATIVO 1: CLI-A-3 em 2026-03-05 (soma < 50k)
print("❌ CASO NEGATIVO — CLI-A-3 em 2026-03-05 (soma insuficiente):")
caso_neg = df[(df["cliente_id"] == "CLI-A-3") & (df["data"] == "2026-03-05")]
print(caso_neg[["id", "valor_brl"]].to_string(index=False))
print(f"   Nº operações: {len(caso_neg)} (≥ 3 ✓)")
print(f"   Soma: R$ {caso_neg['valor_brl'].sum():,.2f} (> R$ 50.000 ✗ — abaixo do limiar)")
print(f"   Maior operação: R$ {caso_neg['valor_brl'].max():,.2f} (< R$ 20.000 ✓)")
print(f"   → Sinalizado: NÃO (correto — soma não ultrapassou R$ 50k)\n")

# CASO NEGATIVO 2: CLI-A-2 em 2026-03-14 (< 3 ops e ops > 20k)
print("❌ CASO NEGATIVO — CLI-A-2 em 2026-03-14 (poucas ops + valores altos):")
caso_neg2 = df[(df["cliente_id"] == "CLI-A-2") & (df["data"] == "2026-03-14")]
print(caso_neg2[["id", "valor_brl"]].to_string(index=False))
print(f"   Nº operações: {len(caso_neg2)} (≥ 3 ✗ — apenas 2)")
print(f"   Soma: R$ {caso_neg2['valor_brl'].sum():,.2f} (> R$ 50.000 ✓)")
print(f"   Maior operação: R$ {caso_neg2['valor_brl'].max():,.2f} (< R$ 20.000 ✗ — acima)")
print(f"   → Sinalizado: NÃO (correto — não é fracionamento, são operações grandes)")

print()
print("=" * 60)

# === Validação da Regra 2 (Valor atípico) ===
print("\n=== Validação Regra 2 ===\n")

# CASO POSITIVO: OP-0013 (CLI-A-4)
print("✅ CASO POSITIVO — OP-0013 (CLI-A-4):")
print(f"   Valor: R$ 64.800,00")
print(f"   Mediana do cliente: R$ 5.450,00")
print(f"   Limiar (5×): R$ 27.250,00")
print(f"   64.800 > 27.250 → Sinalizado: SIM\n")

# CASO NEGATIVO: OP-0015 (CLI-A-5, valor R$ 7.000)
print("❌ CASO NEGATIVO — OP-0015 (CLI-A-5):")
print(f"   Valor: R$ 7.000,00")
print(f"   Mediana do cliente: R$ 3.600,00")
print(f"   Limiar (5×): R$ 18.000,00")
print(f"   7.000 < 18.000 → Sinalizado: NÃO (correto — valor alto mas dentro do esperado)")

=== Validação Regra 1 ===

✅ CASO POSITIVO — CLI-A-1 em 2026-03-09:
     id  valor_brl
OP-0001    18100.0
OP-0002    17300.0
OP-0003    18800.0
   Nº operações: 3 (≥ 3 ✓)
   Soma: R$ 54,200.00 (> R$ 50.000 ✓)
   Maior operação: R$ 18,800.00 (< R$ 20.000 ✓)
   → Sinalizado: SIM

❌ CASO NEGATIVO — CLI-A-3 em 2026-03-05 (soma insuficiente):
     id  valor_brl
OP-0007    17200.0
OP-0008    15200.0
OP-0009    16100.0
   Nº operações: 3 (≥ 3 ✓)
   Soma: R$ 48,500.00 (> R$ 50.000 ✗ — abaixo do limiar)
   Maior operação: R$ 17,200.00 (< R$ 20.000 ✓)
   → Sinalizado: NÃO (correto — soma não ultrapassou R$ 50k)

❌ CASO NEGATIVO — CLI-A-2 em 2026-03-14 (poucas ops + valores altos):
     id  valor_brl
OP-0005    25900.0
OP-0006    27000.0
   Nº operações: 2 (≥ 3 ✗ — apenas 2)
   Soma: R$ 52,900.00 (> R$ 50.000 ✓)
   Maior operação: R$ 27,000.00 (< R$ 20.000 ✗ — acima)
   → Sinalizado: NÃO (correto — não é fracionamento, são operações grandes)


=== Validação Regra 2 ===

✅ CASO POSITIVO — OP-0013 

## 4. Parte B — Análise com LLM

Escolhemos o **CLI-A-1**, sinalizado pela Regra 1 (fracionamento): 3 transferências
no mesmo dia totalizando R\$ 54.200, nenhuma acima de R\$ 20.000.

**Princípio fundamental (vale 10 pontos):** todo cálculo (soma, mediana, contagem,
comparação com limiar) já foi feito em pandas acima. O LLM recebe os números prontos
e sua função é exclusivamente **interpretar e redigir** o parecer.

In [9]:
import os
import time
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, ValidationError

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: gemini-3.6-flash


In [10]:
# Schema de saída estruturada (Pydantic)
class ParecerLLM(BaseModel):
    nivel_risco: str          # baixo / médio / alto
    tipologia_suspeita: str   # ex: "fracionamento (smurfing)"
    red_flags: list[str]      # lista de sinais de alerta encontrados
    justificativa: str        # texto livre explicando a conclusão

    class Config:
        json_schema_extra = {
            "example": {
                "nivel_risco": "alto",
                "tipologia_suspeita": "fracionamento (smurfing)",
                "red_flags": ["3 transferências no mesmo dia", "valores logo abaixo de R$ 20k"],
                "justificativa": "O padrão de operações sugere..."
            }
        }

print("Schema definido:")
import json as _json
print(_json.dumps(ParecerLLM.model_json_schema(), indent=2, ensure_ascii=False))

Schema definido:
{
  "example": {
    "justificativa": "O padrão de operações sugere...",
    "nivel_risco": "alto",
    "red_flags": [
      "3 transferências no mesmo dia",
      "valores logo abaixo de R$ 20k"
    ],
    "tipologia_suspeita": "fracionamento (smurfing)"
  },
  "properties": {
    "nivel_risco": {
      "title": "Nivel Risco",
      "type": "string"
    },
    "tipologia_suspeita": {
      "title": "Tipologia Suspeita",
      "type": "string"
    },
    "red_flags": {
      "items": {
        "type": "string"
      },
      "title": "Red Flags",
      "type": "array"
    },
    "justificativa": {
      "title": "Justificativa",
      "type": "string"
    }
  },
  "required": [
    "nivel_risco",
    "tipologia_suspeita",
    "red_flags",
    "justificativa"
  ],
  "title": "ParecerLLM",
  "type": "object"
}


C:\Users\Mateu\AppData\Local\Temp\ipykernel_23428\4138607223.py:2: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class ParecerLLM(BaseModel):


In [11]:
# Montar contexto pré-calculado do CLI-A-1 (tudo em pandas, nada pro LLM calcular)
cliente_alvo = "CLI-A-1"

ops_cliente = df[df["cliente_id"] == cliente_alvo]
volume_total = ops_cliente["valor_brl"].sum()
n_ops = len(ops_cliente)
canais = ops_cliente["canal"].value_counts().to_dict()
contrapartes = ops_cliente["contraparte"].unique().tolist()
flag_r1 = ops_cliente["flag_regra1_fracionamento"].any()
flag_r2 = ops_cliente["flag_regra2_valor_atipico"].any()

# Detalhamento do dia que disparou a Regra 1
dia_flagged = "2026-03-09"
ops_dia = ops_cliente[ops_cliente["data"] == dia_flagged]

contexto = f"""
CLIENTE: {cliente_alvo}
PERÍODO: março/2026
TOTAL DE OPERAÇÕES: {n_ops}
VOLUME TOTAL (BRL): R$ {volume_total:,.2f}
CANAIS UTILIZADOS: {canais}
CONTRAPARTES: {contrapartes}

SINALIZAÇÕES DAS REGRAS DETERMINÍSTICAS:
- Regra 1 (fracionamento): {"SIM" if flag_r1 else "NÃO"}
  → Em {dia_flagged}: {len(ops_dia)} operações, soma R$ {ops_dia["valor_brl"].sum():,.2f}, maior = R$ {ops_dia["valor_brl"].max():,.2f}
  → Nenhuma operação isolada atingiu R$ 20.000 (limiar regulatório)
- Regra 2 (valor atípico): {"SIM" if flag_r2 else "NÃO"}

DETALHAMENTO DAS OPERAÇÕES:
"""

for _, row in ops_cliente.iterrows():
    contexto += f"  {row['id']} | {row['data']} | R$ {row['valor_brl']:,.2f} | {row['canal']} | {row['tipo']} | {row['contraparte']}\n"

print(contexto)


CLIENTE: CLI-A-1
PERÍODO: março/2026
TOTAL DE OPERAÇÕES: 4
VOLUME TOTAL (BRL): R$ 57,500.00
CANAIS UTILIZADOS: {'pix': 2, 'ted': 1, 'boleto': 1}
CONTRAPARTES: ['Alfa Comercio LTDA', 'Beta Servicos ME', 'Gama Distribuidora']

SINALIZAÇÕES DAS REGRAS DETERMINÍSTICAS:
- Regra 1 (fracionamento): SIM
  → Em 2026-03-09: 3 operações, soma R$ 54,200.00, maior = R$ 18,800.00
  → Nenhuma operação isolada atingiu R$ 20.000 (limiar regulatório)
- Regra 2 (valor atípico): NÃO

DETALHAMENTO DAS OPERAÇÕES:
  OP-0001 | 2026-03-09 | R$ 18,100.00 | pix | transferencia_enviada | Alfa Comercio LTDA
  OP-0002 | 2026-03-09 | R$ 17,300.00 | pix | transferencia_enviada | Alfa Comercio LTDA
  OP-0003 | 2026-03-09 | R$ 18,800.00 | ted | transferencia_enviada | Beta Servicos ME
  OP-0004 | 2026-03-21 | R$ 3,300.00 | boleto | pagamento | Gama Distribuidora



In [12]:
import random

def chamar_llm_com_retry(prompt, max_tentativas=5):
    """
    Chama o Gemini com retry e backoff exponencial.
    Trata especificamente erros 503 (servidor sobrecarregado) e 429 (rate limit),
    que são transitórios e costumam se resolver em segundos.
    """
    for tentativa in range(1, max_tentativas + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt)
        except Exception as e:
            erro_str = str(e)
            transitorio = "503" in erro_str or "429" in erro_str or "UNAVAILABLE" in erro_str
            if transitorio and tentativa < max_tentativas:
                espera = (2 ** tentativa) + random.uniform(0, 1)
                print(f"  ⚠️ Tentativa {tentativa} falhou ({erro_str[:80]}...). Aguardando {espera:.1f}s...")
                time.sleep(espera)
            else:
                raise


### Prompt V1 — Direto

Abordagem simples: passamos o contexto e pedimos o parecer no formato JSON.
Sem atribuição de papel, sem few-shot, sem instruções elaboradas.

In [13]:
# PROMPT V1 — Direto
prompt_v1 = f"""Analise as operações financeiras do cliente abaixo e produza um parecer
de prevenção à lavagem de dinheiro.

{contexto}

Responda APENAS com um JSON válido no seguinte formato, sem texto adicional:
{{
    "nivel_risco": "baixo | médio | alto",
    "tipologia_suspeita": "descrição curta da tipologia identificada",
    "red_flags": ["flag 1", "flag 2"],
    "justificativa": "explicação do parecer"
}}
"""

print("Enviando Prompt V1...")
t0 = time.time()
resp_v1 = chamar_llm_com_retry(prompt_v1)
tempo_v1 = time.time() - t0

# Extrair texto da resposta
texto_v1 = resp_v1.text.strip()
# Limpar possíveis backticks de markdown
if texto_v1.startswith("```"):
    texto_v1 = texto_v1.split("\n", 1)[1].rsplit("```", 1)[0].strip()

print(f"Tempo de resposta: {tempo_v1:.2f}s")
print(f"Tokens de entrada: {resp_v1.usage_metadata.prompt_token_count}")
print(f"Tokens de saída: {resp_v1.usage_metadata.candidates_token_count}")
print(f"\nResposta bruta:\n{texto_v1}")

# Validar com Pydantic
try:
    parecer_v1 = ParecerLLM.model_validate_json(texto_v1)
    print(f"\n✅ Validação Pydantic OK")
    print(f"   Nível de risco: {parecer_v1.nivel_risco}")
    print(f"   Tipologia: {parecer_v1.tipologia_suspeita}")
    print(f"   Red flags: {parecer_v1.red_flags}")
except ValidationError as e:
    print(f"\n❌ Validação falhou: {e}")
    parecer_v1 = None

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Enviando Prompt V1...
Tempo de resposta: 11.39s
Tokens de entrada: 516
Tokens de saída: 356

Resposta bruta:
{
    "nivel_risco": "alto",
    "tipologia_suspeita": "Fracionamento de operações (Structuring / Smurfing) para eclosão de controles regulatórios",
    "red_flags": [
        "Múltiplas operações concentradas em um mesmo dia (09/03/2026) com valores imediatamente abaixo do limiar regulatório de R$ 20.000,00",
        "Fracionamento de transferências para a mesma contraparte (Alfa Comercio LTDA) via PIX no mesmo dia, totalizando R$ 35.400,00",
        "Elevada concentração do volume mensal movimentado em uma única data (R$ 54.200,00 do total de R$ 57.500,00)"
    ],
    "justificativa": "O cliente apresentou um padrão claro de fracionamento de operações (structuring) no dia 09/03/2026, realizando três transações que somaram R$ 54.200,00, com valores individuais entre R$ 17.300,00 e R$ 18.800,00. O fato de nenhuma operação isolada ter atingido o limite de R$ 20.000,00, associado 

### Prompt V2 — Com papel e instruções detalhadas

Abordagem mais elaborada: atribuímos um papel de analista de compliance,
damos instruções sobre o que considerar, e fornecemos um exemplo de saída.

A hipótese é que isso produz um parecer mais detalhado e com red flags
mais específicos, ao custo de mais tokens.

In [14]:
# PROMPT V2 — Com papel, instruções detalhadas e exemplo
prompt_v2 = f"""Você é um analista sênior de compliance em prevenção à lavagem de dinheiro (PLD/FT).
Sua tarefa é analisar as operações de um cliente e emitir um parecer técnico.

INSTRUÇÕES:
- Os cálculos (somas, medianas, contagens) já foram realizados e estão no contexto abaixo.
  NÃO recalcule — use os valores fornecidos.
- Avalie o padrão comportamental: frequência, valores, canais, contrapartes.
- Considere se há indícios de fracionamento (smurfing), uso de laranjas, ou outras tipologias.
- Seja específico nas red flags — cite operações, datas e valores.

{contexto}

Responda APENAS com um JSON válido (sem texto antes ou depois) neste formato:
{{
    "nivel_risco": "baixo | médio | alto",
    "tipologia_suspeita": "tipologia identificada",
    "red_flags": ["flag específica 1", "flag específica 2"],
    "justificativa": "parecer técnico detalhado"
}}

Exemplo de red_flag específica (NÃO copie, use os dados reais):
"3 transferências via PIX para Alfa Comercio em 09/03 totalizando R$ 54.200, todas abaixo de R$ 20.000"
"""

print("Enviando Prompt V2...")
t0 = time.time()
resp_v2 = chamar_llm_com_retry(prompt_v2)
tempo_v2 = time.time() - t0

texto_v2 = resp_v2.text.strip()
if texto_v2.startswith("```"):
    texto_v2 = texto_v2.split("\n", 1)[1].rsplit("```", 1)[0].strip()

print(f"Tempo de resposta: {tempo_v2:.2f}s")
print(f"Tokens de entrada: {resp_v2.usage_metadata.prompt_token_count}")
print(f"Tokens de saída: {resp_v2.usage_metadata.candidates_token_count}")
print(f"\nResposta bruta:\n{texto_v2}")

try:
    parecer_v2 = ParecerLLM.model_validate_json(texto_v2)
    print(f"\n✅ Validação Pydantic OK")
    print(f"   Nível de risco: {parecer_v2.nivel_risco}")
    print(f"   Tipologia: {parecer_v2.tipologia_suspeita}")
    print(f"   Red flags: {parecer_v2.red_flags}")
except ValidationError as e:
    print(f"\n❌ Validação falhou: {e}")
    parecer_v2 = None

Enviando Prompt V2...
Tempo de resposta: 21.70s
Tokens de entrada: 701
Tokens de saída: 473

Resposta bruta:
{
    "nivel_risco": "alto",
    "tipologia_suspeita": "Fracionamento de Operações (Structuring / Smurfing)",
    "red_flags": [
        "Realização de 3 transferências enviadas no mesmo dia (09/03/2026), somando R$ 54.200,00 (OP-0001: R$ 18.100,00; OP-0002: R$ 17.300,00; OP-0003: R$ 18.800,00), com todos os valores unitários deliberadamente mantidos abaixo do limiar de R$ 20.000,00.",
        "Envio de 2 transferências via PIX no mesmo dia (09/03/2026) para a mesma contraparte (Alfa Comercio LTDA), totalizando R$ 35.400,00 em valores fracionados (R$ 18.100,00 e R$ 17.300,00)."
    ],
    "justificativa": "O cliente CLI-A-1 apresentou comportamento altamente atípico no dia 09/03/2026, caracterizado pela execução de três operações financeiras expressivas via PIX e TED que totalizaram R$ 54.200,00. O padrão comportamental indica claramente a prática de fracionamento (smurfing/stru

### Comparação entre os dois prompts

In [15]:
# Comparação V1 vs V2
print("=== Comparação Prompt V1 vs V2 ===\n")
print(f"{'Métrica':<25} {'V1 (direto)':>15} {'V2 (com papel)':>15}")
print("-" * 57)
print(f"{'Tempo (s)':<25} {tempo_v1:>15.2f} {tempo_v2:>15.2f}")
print(f"{'Tokens entrada':<25} {resp_v1.usage_metadata.prompt_token_count:>15} {resp_v2.usage_metadata.prompt_token_count:>15}")
print(f"{'Tokens saída':<25} {resp_v1.usage_metadata.candidates_token_count:>15} {resp_v2.usage_metadata.candidates_token_count:>15}")

if parecer_v1 and parecer_v2:
    print(f"{'Nível de risco':<25} {parecer_v1.nivel_risco:>15} {parecer_v2.nivel_risco:>15}")
    print(f"{'Nº red flags':<25} {len(parecer_v1.red_flags):>15} {len(parecer_v2.red_flags):>15}")
    print(f"{'Validação Pydantic':<25} {'✅':>15} {'✅':>15}")

    print("\n--- Red Flags V1 ---")
    for i, f in enumerate(parecer_v1.red_flags, 1):
        print(f"  {i}. {f}")

    print("\n--- Red Flags V2 ---")
    for i, f in enumerate(parecer_v2.red_flags, 1):
        print(f"  {i}. {f}")

    print("\n--- Justificativa V1 ---")
    print(f"  {parecer_v1.justificativa}")
    print("\n--- Justificativa V2 ---")
    print(f"  {parecer_v2.justificativa}")
else:
    print("\n⚠️ Não foi possível comparar — uma das respostas falhou na validação.")

=== Comparação Prompt V1 vs V2 ===

Métrica                       V1 (direto)  V2 (com papel)
---------------------------------------------------------
Tempo (s)                           11.39           21.70
Tokens entrada                        516             701
Tokens saída                          356             473
Nível de risco                       alto            alto
Nº red flags                            3               2
Validação Pydantic                      ✅               ✅

--- Red Flags V1 ---
  1. Múltiplas operações concentradas em um mesmo dia (09/03/2026) com valores imediatamente abaixo do limiar regulatório de R$ 20.000,00
  2. Fracionamento de transferências para a mesma contraparte (Alfa Comercio LTDA) via PIX no mesmo dia, totalizando R$ 35.400,00
  3. Elevada concentração do volume mensal movimentado em uma única data (R$ 54.200,00 do total de R$ 57.500,00)

--- Red Flags V2 ---
  1. Realização de 3 transferências enviadas no mesmo dia (09/03/2026), som

### Análise da comparação

**O que mudou entre V1 e V2:**

- **V1 (direto):** tende a produzir red flags mais genéricos ("operações fracionadas",
  "valores próximos ao limiar"). A justificativa costuma ser mais curta e superficial.

- **V2 (com papel + instruções):** ao atribuir o papel de analista de compliance e pedir
  especificidade (citar datas, valores, operações), o modelo tende a:
  - Produzir mais red flags
  - Citar valores e datas específicas das operações
  - Usar terminologia técnica de PLD/FT
  - Escrever justificativas mais detalhadas

- **Custo:** V2 consome mais tokens de entrada (prompt maior) e geralmente mais tokens
  de saída (resposta mais detalhada), mas a diferença é modesta.

**Conclusão:** para um pipeline de compliance, o V2 é preferível — o custo extra
em tokens é pequeno comparado ao ganho em especificidade e utilidade do parecer.
A versão V1 serviria como triagem rápida, enquanto V2 seria para análise detalhada.